## Importación de librerías

In [6]:
import sys
import gurobipy as gp
from gurobipy import GRB
import time
import csv
import math
import random

print('Python  :', sys.version.split()[0])
print('Gurobi  :', '.'.join(map(str, gp.gurobi.version())))

Python  : 3.12.10
Gurobi  : 13.0.3


### 1. Generación de Datos (Mock Data para probar)

In [7]:
import random
import math

def generar_instancia_desde_cvrplib(nombre_instancia, datos_clientes, n_clientes=15, tipo_estres="capacidad"):
    """
    Genera un diccionario de datos compatible con el ConsistentVRPFramework.
    """
    N_C = list(range(1, n_clientes + 1))
    N = list(range(n_clientes + 2)) # 0 es inicio, n+1 es fin
    D = [1, 2, 3] # 3 días de planificación
    R = [1, 2, 3] # 3 rutas por día

    # Extraer coordenadas del depósito (índice 0)
    coord_panaderia = (datos_clientes[0]['x'], datos_clientes[0]['y'])
    coordenadas = {0: coord_panaderia, n_clientes + 1: coord_panaderia}
    
    for i in N_C:
        coordenadas[i] = (datos_clientes[i]['x'], datos_clientes[i]['y'])

    # Conjunto de arcos estrictos
    A = [(i, j) for i in N for j in N if i != j and j != 0 and i != (n_clientes + 1)]

    td = {}
    tt = {}
    for (i, j) in A:
        dist = math.sqrt((coordenadas[i][0] - coordenadas[j][0])**2 + (coordenadas[i][1] - coordenadas[j][1])**2)
        td[i, j] = dist
        tt[i, j] = dist # Tiempo proporcional a la distancia

    q = {}
    st = {}
    a = {}
    b = {}
    N_TW = []
    
    demanda_total_base = sum(datos_clientes[i]['demanda'] for i in N_C)
    
    # Ajuste de capacidad según el tipo de estrés seleccionado
    if tipo_estres == "capacidad":
        Q = math.ceil((demanda_total_base / 0.85) / len(R)) 
    else:
        Q = math.ceil((demanda_total_base / 0.50) / len(R))

    for i in N_C:
        demanda_base = datos_clientes[i]['demanda']
        
        # Generar consistencia en los 3 días
        for t in D:
            if tipo_estres == "capacidad":
                if t == 1: q[i, t] = demanda_base
                elif t == 2: q[i, t] = math.ceil(demanda_base * 1.3) # Peak de demanda
                else: q[i, t] = math.ceil(demanda_base * 0.8)
            else:
                 q[i, t] = demanda_base 
                 
            st[i, t] = 10 
            
        # Asignación de ventanas de tiempo
        tw_inicio = datos_clientes[i]['inicio_tw']
        tw_fin = datos_clientes[i]['fin_tw']
        
        if (tw_fin - tw_inicio) < 1000: 
            N_TW.append(i)
            a[i] = tw_inicio
            if tipo_estres == "tiempo":
                b[i] = tw_inicio + (tw_fin - tw_inicio) * 0.3 # Aprieta la ventana
            else:
                b[i] = tw_fin

    data_instancia = {
        'nombre': nombre_instancia,
        'A': A, 'N_C': N_C, 'N': N, 'N_TW': N_TW, 'R': R, 'D': D,
        'td': td, 'tt': tt, 'q': q, 'st': st,
        'a': a, 'b': b,
        'a_W': datos_clientes[0]['inicio_tw'], 
        'b_W': datos_clientes[0]['fin_tw'], 
        'Q': Q, 
        'M': 5000 
    }
    return data_instancia

# === DATOS SIMULADOS BASADOS EN SOLOMON C101 ===
datos_base_cvrplib = [
    {'x': 40, 'y': 50, 'demanda': 0, 'inicio_tw': 0, 'fin_tw': 1236}, # Depósito
    {'x': 45, 'y': 68, 'demanda': 10, 'inicio_tw': 912, 'fin_tw': 967},
    {'x': 45, 'y': 70, 'demanda': 30, 'inicio_tw': 825, 'fin_tw': 870},
    {'x': 42, 'y': 66, 'demanda': 10, 'inicio_tw': 65, 'fin_tw': 146},
    {'x': 42, 'y': 68, 'demanda': 10, 'inicio_tw': 727, 'fin_tw': 782},
    {'x': 42, 'y': 65, 'demanda': 10, 'inicio_tw': 15, 'fin_tw': 67},
    {'x': 40, 'y': 69, 'demanda': 20, 'inicio_tw': 621, 'fin_tw': 702},
    {'x': 40, 'y': 66, 'demanda': 20, 'inicio_tw': 170, 'fin_tw': 225},
    {'x': 38, 'y': 68, 'demanda': 20, 'inicio_tw': 255, 'fin_tw': 324},
    {'x': 38, 'y': 70, 'demanda': 10, 'inicio_tw': 534, 'fin_tw': 605},
    {'x': 35, 'y': 66, 'demanda': 10, 'inicio_tw': 357, 'fin_tw': 410},
    {'x': 35, 'y': 69, 'demanda': 10, 'inicio_tw': 448, 'fin_tw': 505},
    {'x': 25, 'y': 85, 'demanda': 20, 'inicio_tw': 652, 'fin_tw': 721},
    {'x': 22, 'y': 75, 'demanda': 30, 'inicio_tw': 30, 'fin_tw': 92},
    {'x': 22, 'y': 85, 'demanda': 10, 'inicio_tw': 567, 'fin_tw': 620},
    {'x': 20, 'y': 80, 'demanda': 40, 'inicio_tw': 384, 'fin_tw': 429}
]

# Generación de la instancia base
data_instancia = generar_instancia_desde_cvrplib(
    nombre_instancia="C101_Urbana_Concepcion", 
    datos_clientes=datos_base_cvrplib, 
    n_clientes=15, 
    tipo_estres="tiempo"
)

print(f"Instancia '{data_instancia['nombre']}' generada correctamente. Q = {data_instancia['Q']}")

Instancia 'C101_Urbana_Concepcion' generada correctamente. Q = 174


### 2. Definición del Framework Matemático

In [ ]:
class ConsistentVRPFramework:
    def __init__(self, data):
        self.data = data
        self.model = None
        self.X = {}
        self.Z = {}
        self.W = {}
        self.dW = {}
        self.rW = {}
        
    def build_model(self):
        self.model = gp.Model("CVRP_Consistent_Panaderia")
        
        N_C, N, N_TW, R, D = self.data['N_C'], self.data['N'], self.data['N_TW'], self.data['R'], self.data['D']
        td, tt, q, st = self.data['td'], self.data['tt'], self.data['q'], self.data['st']
        a, b, a_W, b_W, Q, M = self.data['a'], self.data['b'], self.data['a_W'], self.data['b_W'], self.data['Q'], self.data['M']
        
        A = self.data['A']

        self.X = self.model.addVars(A, D, R, vtype=GRB.BINARY, name="X")
        self.Z = self.model.addVars(N_C, R, vtype=GRB.BINARY, name="Z")
        self.W = self.model.addVars(N_C, D, vtype=GRB.CONTINUOUS, lb=0.0, name="W")
        self.dW = self.model.addVars(R, vtype=GRB.CONTINUOUS, lb=0.0, name="dW")
        self.rW = self.model.addVars(R, D, vtype=GRB.CONTINUOUS, lb=0.0, name="rW")

        self.model.setObjective(gp.quicksum(td[i, j] * self.X[i, j, t, r] for (i, j) in A for t in D for r in R), GRB.MINIMIZE)

        for i in N_C:
            self.model.addConstr(gp.quicksum(self.Z[i, r] for r in R) == 1, name=f"AsignacionUnica_{i}")

        # CAMBIO 2: Iteración directa sobre los arcos entrantes y salientes para mejor eficiencia
        for j in N_C:
            for r in R:
                for t in D:
                    # Sumar solo los nodos 'i' que tienen un arco válido hacia 'j'
                    self.model.addConstr(gp.quicksum(self.X[i, j, t, r] for i in N if (i, j) in A) == self.Z[j, r], name=f"FlujoEntrada_{j}_{r}_{t}")
        
        for i in N_C:
            for r in R:
                for t in D:
                    # Sumar solo los nodos 'j' hacia donde hay un arco válido desde 'i'
                    self.model.addConstr(gp.quicksum(self.X[i, j, t, r] for j in N if (i, j) in A) == self.Z[i, r], name=f"FlujoSalida_{i}_{r}_{t}")

        for r in R:
            for t in D:
                 self.model.addConstr(gp.quicksum(self.X[0, j, t, r] for j in N if (0, j) in A) == 1, name=f"SalidaDep_{r}_{t}")
                 
        for r in R:
            for t in D:
                for i in N_C:
                    for j in N_C:
                        if i != j and (i, j) in A:
                            self.model.addConstr(self.W[i, t] + st[i, t] + tt[i, j] <= self.W[j, t] + M * (1 - self.X[i, j, t, r]), name=f"BigM_{i}_{j}_{r}_{t}")

        for i in N_TW:
            for t in D:
                self.model.addConstr(self.W[i, t] >= a[i], name=f"TW_Inicio_{i}_{t}")
                self.model.addConstr(self.W[i, t] <= b[i], name=f"TW_Fin_{i}_{t}")

        n_plus_1 = max(N)
        for j in N_C:
            for r in R:
                for t in D:
                    self.model.addConstr(self.dW[r] + tt[0, j] <= self.W[j, t] + M * (1 - self.X[0, j, t, r]), name=f"CoordSalida_{j}_{r}_{t}")
        for i in N_C:
            for r in R:
                for t in D:
                    self.model.addConstr(self.W[i, t] + st[i, t] + tt[i, n_plus_1] <= self.rW[r, t] + M * (1 - self.X[i, n_plus_1, t, r]), name=f"CoordRet_{i}_{r}_{t}")

        for r in R:
            self.model.addConstr(self.dW[r] >= a_W, name=f"Ap_Dep_{r}")
            for t in D:
                self.model.addConstr(self.rW[r, t] <= b_W, name=f"Ci_Dep_{r}_{t}")
                self.model.addConstr(gp.quicksum(q[i, t] * self.Z[i, r] for i in N_C) <= Q, name=f"Capacidad_{r}_{t}")  

        self.model.update()

    def set_base_params(self):
        self.model.setParam('Threads', 1)
        self.model.setParam('TimeLimit', 600)
        self.model.setParam('OutputFlag', 0)

    def run_relaxation(self):
        self.set_base_params()
        relaxed = self.model.relax()
        relaxed.optimize()
        return relaxed.ObjVal if relaxed.Status == GRB.OPTIMAL else None

    def run_configuration(self, config_name, params_dict):
        self.model.reset() 
        self.set_base_params()
        for p, v in params_dict.items():
            self.model.setParam(p, v)
            
        start = time.time()
        self.model.optimize()
        t = time.time() - start
        
        if self.model.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT] and self.model.SolCount > 0:
            return self.model.ObjVal, self.model.MIPGap * 100, t
        return None, None, t

    def run_tuning(self, tune_time_limit=300):
            print("\nEjecutando Gurobi Tune...")
            self.model.reset()
            self.set_base_params()
            self.model.setParam('TuneTimeLimit', tune_time_limit)
            
            # El tiempo total incluye la fase de tuneo y la optimización final
            start = time.time() 
            self.model.tune()

            if self.model.TuneResultCount > 0:
                self.model.getTuneResult(0)
                self.model.optimize()
                t = time.time() - start
                return self.model.ObjVal, self.model.MIPGap * 100, t
            
            t = time.time() - start
            return None, None, t

    def run_sensitivity_analysis(self, filename="sensibilidad_gurobi.csv"):
        # Seguro: Verificar que tengamos una solución antes de intentar fijar variables
        if self.model.SolCount == 0:
            print("\n[!] Advertencia: No hay solución factible en memoria.")
            print("[!] Gurobi no puede hacer análisis de sensibilidad sin una solución. Aumenta el TimeLimit.")
            return

        fixed_model = self.model.fixed()
        fixed_model.setParam('QCPDual', 1) 
        fixed_model.setParam('OutputFlag', 0)
        fixed_model.optimize()

        if fixed_model.Status == GRB.OPTIMAL:
            with open(filename, mode='w', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow(["Restriccion", "Precio_Sombra_Pi", "SARHSLow", "SARHSUp"])
                for r in self.data['R']:
                    for t in self.data['D']:
                        constr = fixed_model.getConstrByName(f"Capacidad_{r}_{t}")
                        if constr:
                            writer.writerow([f"Cap_{r}_{t}", f"{constr.Pi:.4f}", f"{constr.SARHSLow:.2f}", f"{constr.SARHSUp:.2f}"])
            print(f"Análisis de sensibilidad guardado exitosamente en {filename}")

    def exportar_resultados_csv(self, z_LP, res_milp, filename="resultados_gurobi.csv"):
        with open(filename, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(["Configuracion", "Valor_Objetivo", "Gap_%", "Tiempo_s", "Brecha_Integralidad_%"])
            writer.writerow(["LP Relax", f"{z_LP:.4f}" if z_LP else "N/A", "-", "-", "-"])
            
            z_MILP_base = res_milp.get("Baseline", (None, None, None))[0]
            for config, (obj, gap, run_time) in res_milp.items():
                brecha = f"{((z_MILP_base - z_LP) / z_MILP_base * 100):.2f}" if config == "Baseline" and z_LP and z_MILP_base else "-"
                writer.writerow([config, f"{obj:.4f}" if obj else "N/A", f"{gap:.2f}" if gap else "-", f"{run_time:.2f}", brecha])

### 3. Construcción y Ejecución de Experimentos

In [ ]:
# Inicializar y construir modelo
framework = ConsistentVRPFramework(data_instancia) # Recuerda hacerlo con 2 instancias distintas luego
framework.build_model()

print("1. Calculando Relajación Continua...")
z_LP = framework.run_relaxation()
print(f"Valor LP: {z_LP:.2f}" if z_LP else "Sin solución LP")

# Configuraciones a testear
configs = {
    "Baseline": {},
    "No_Cuts": {'Cuts': 0},
    "No_Presolve": {'Presolve': 0},
    "MIPFocus_1": {'MIPFocus': 1},
    "MIPFocus_3": {'MIPFocus': 3},
    "No_Heuristics": {'Heuristics': 0}
}

resultados_milp = {}
print("\n2. Ejecutando Configuraciones MILP...")
for nombre, params in configs.items():
    print(f"Corriendo {nombre}...")
    obj, gap, t = framework.run_configuration(nombre, params)
    resultados_milp[nombre] = (obj, gap, t)
    print(f" -> Obj: {obj}, Gap: {gap}%, Tiempo: {t:.2f}s")

print("\n3. Ejecutando Ajuste Automático (Tuning)...")
obj_tune, gap_tune, t_tune = framework.run_tuning(tune_time_limit=300)
resultados_milp["Tuning_Gurobi"] = (obj_tune, gap_tune, t_tune)
print(f" -> Obj: {obj_tune}, Gap: {gap_tune}%, Tiempo total: {t_tune:.2f}s")

# Analisis de Sensibilidad y Exportación
print("\n4. Preparando Análisis de Sensibilidad...")
# Aseguramos un incumbente con la configuración base antes de fijar
framework.run_configuration("Baseline", {'TimeLimit': 600}) 
framework.run_sensitivity_analysis("sensibilidad_instancia.csv")

framework.exportar_resultados_csv(z_LP, resultados_milp, "resultados_instancia.csv")
print("\n¡Experimentos de la Instancia finalizados y exportados a CSV!")

1. Calculando Relajación Continua...
Set parameter Threads to value 1
Set parameter TimeLimit to value 10
Valor LP: 121.44

2. Ejecutando Configuraciones MILP...
Corriendo Baseline...
 -> Obj: 426.4298090358611, Gap: 0.0%, Tiempo: 0.02s
Corriendo No_Cuts...
 -> Obj: 426.4298090358611, Gap: 0.0%, Tiempo: 0.03s
Corriendo No_Presolve...
 -> Obj: 426.4298090358611, Gap: 0.0%, Tiempo: 0.03s
Corriendo MIPFocus_1...
 -> Obj: 426.4298090358611, Gap: 0.0%, Tiempo: 0.04s
Corriendo MIPFocus_3...
 -> Obj: 426.4298090358611, Gap: 0.0%, Tiempo: 0.04s
Corriendo No_Heuristics...
 -> Obj: 426.4298090358611, Gap: 0.0%, Tiempo: 0.02s

3. Ejecutando Ajuste Automático (Tuning)...

Ejecutando Gurobi Tune...
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-10300H CPU @ 2.50GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 1 threads

Non-default parameters:
Heuristics  0
MIPFocus  3
Cuts  0


### 4. Visualización de resultados

In [13]:
import pandas as pd

print("--- Resultados de Configuraciones ---")
df_resultados = pd.read_csv("resultados_instancia.csv")
display(df_resultados)

print("\n--- Análisis de Sensibilidad (Capacidad) ---")
df_sensibilidad = pd.read_csv("sensibilidad_instancia.csv")
display(df_sensibilidad.head(6)) # Muestra las primeras 6 filas

--- Resultados de Configuraciones ---


,Configuracion,Valor_Objetivo,Gap_%,Tiempo_s,Brecha_Integralidad_%
0,LP Relax,121.4356,-,-,-
1,Baseline,426.4298,-,0.02,71.52
2,No_Cuts,426.4298,-,0.03,-
3,No_Presolve,426.4298,-,0.03,-
4,MIPFocus_1,426.4298,-,0.04,-
5,MIPFocus_3,426.4298,-,0.04,-
6,No_Heuristics,426.4298,-,0.02,-
7,Tuning_Gurobi,426.4298,-,10.10,-



--- Análisis de Sensibilidad (Capacidad) ---


,Restriccion,Precio_Sombra_Pi,SARHSLow,SARHSUp
0,Cap_1_1,0.0,100.0,inf
1,Cap_1_2,0.0,100.0,inf
2,Cap_1_3,0.0,100.0,inf
3,Cap_2_1,0.0,0.0,inf
4,Cap_2_2,0.0,0.0,inf
5,Cap_2_3,0.0,0.0,inf
